# ML-07 — Baseline Action Score and Top-10 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/flyrank-bih/flyrank-ml-internship-starter/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My rule and its reason codes

**Lane:** content decay: which pages should a human review first.

**Data and label:** the starter dataset, with 26,612 labeled pages (3,388 have no prior-30-day baseline and are excluded). `is_decayed` = impressions fell 30% or more, last 30 days versus the prior 30 days (same as ML-03). The label is used only to test signals and grade the queue, never as a rule input.

**Not allowed as rule inputs:** `trend_direction`, `trend_pct`, and every `*_last_30d` and `*_prev_30d` column, because they define the label or sit inside its window.

In [1]:
!test -d repo || git clone --depth 1 https://github.com/flyrank-bih/flyrank-ml-internship-starter.git repo
!cp -r repo/data .

In [2]:
import numpy as np
import pandas as pd

SEED = 42
df = pd.read_csv("data/raw/content_refresh_anonymized.csv")
print("Loaded:", df.shape)

# Same label as ML-03: needs a prior-30-day baseline, then a fall of 30% or more
df = df[df["trend_pct"].notna()].reset_index(drop=True)
df["is_decayed"] = ((df["trend_direction"] == "down") & (df["trend_pct"] <= -30)).astype(int)

# These define the label or sit inside the label window, so no rule may use them
FORBIDDEN = ["trend_direction", "trend_pct", "is_decayed"] + \
            [c for c in df.columns if c.endswith(("_last_30d", "_prev_30d"))]

print("Labeled pages:", f"{len(df):,}", "| decaying share (base rate):", round(df["is_decayed"].mean(), 3))
print("Not allowed as rule inputs:", FORBIDDEN)

Loaded: (30000, 44)
Labeled pages: 26,612 | decaying share (base rate): 0.531
Not allowed as rule inputs: ['trend_direction', 'trend_pct', 'is_decayed', 'impressions_last_30d', 'clicks_last_30d', 'sessions_last_30d', 'impressions_prev_30d', 'clicks_prev_30d', 'sessions_prev_30d']


### Signal 1: staleness (linked to FlyRank's refresh flags)

**Claim:** pages that have gone longer without an update are more likely to be decaying.

**Test:** group labeled pages by `days_since_last_update` into four buckets and compare the decay rate in each bucket with the base rate. `n` is shown for every bucket. A bucket needs at least 50 pages to count.

**Verdict rules (set before looking at the table):**
- **CONFIRMED:** the decay rate rises as pages get staler.
- **OPPOSITE:** the decay rate falls as pages get staler.
- **MIXED:** the decay rate goes up and down across buckets.
- **FALSE:** the buckets are all within 3 percentage points of each other.

In [3]:
bins = [0, 30, 90, 180, np.inf]
labels = ["1-30 days", "31-90 days", "91-180 days", "181+ days"]
bucket = pd.cut(df["days_since_last_update"], bins=bins, labels=labels)

base = df["is_decayed"].mean()
t1 = (df.groupby(bucket, observed=True)["is_decayed"]
        .agg(n="size", decaying="sum", decay_rate="mean").round(3))
t1["lift_vs_base"] = (t1["decay_rate"] / base).round(2)

print(f"Base rate: {base:.3f}")
t1

Base rate: 0.531


,n,decaying,decay_rate,lift_vs_base
days_since_last_update,,,,
1-30 days,17454,9181,0.526,0.99
31-90 days,167,91,0.545,1.03
91-180 days,8858,4791,0.541,1.02
181+ days,133,75,0.564,1.06


**Observed:** base rate 0.531. Decay rate by bucket: 1-30 days 0.526 (n = 17,454), 31-90 days 0.545 (n = 167), 91-180 days 0.541 (n = 8,858), 181+ days 0.564 (n = 133). The n values add up to 26,612.

**Verdict: MIXED.** The rate is not steady across buckets, and the full spread is 3.8 points, above the 3-point FALSE cut. The two large buckets differ by only 1.5 points, so the signal is very weak; the two small buckets are too small to read.

### Signal 2: volume (linked to FlyRank's quick-win logic)

**Claim:** pages with more search impressions are more likely to be decaying, so volume tells us where review effort matters.

**Test:** group labeled pages by `impressions_90d` into five buckets and compare the decay rate in each with the base rate. `n` is shown for every bucket; the floor is 50 pages.

**Verdict rules (set before looking at the table, checked in this order):**
1. **FALSE:** all buckets are within 3 percentage points of each other.
2. **CONFIRMED:** otherwise, the decay rate rises at every step up in volume.
3. **OPPOSITE:** otherwise, the decay rate falls at every step up in volume.
4. **MIXED:** otherwise (it goes up and down).

In [4]:
bins = [0, 100, 500, 2000, 10000, np.inf]
labels = ["<100", "100-499", "500-1,999", "2,000-9,999", "10,000+"]
bucket = pd.cut(df["impressions_90d"], bins=bins, labels=labels, right=False)

t2 = (df.groupby(bucket, observed=True)["is_decayed"]
        .agg(n="size", decaying="sum", decay_rate="mean").round(3))
t2["lift_vs_base"] = (t2["decay_rate"] / base).round(2)

print(f"Base rate: {base:.3f}")
t2

Base rate: 0.531


,n,decaying,decay_rate,lift_vs_base
impressions_90d,,,,
<100,4854,2964,0.611,1.15
100-499,5141,2869,0.558,1.05
"500-1,999",6449,3455,0.536,1.01
"2,000-9,999",6573,3369,0.513,0.97
"10,000+",3595,1481,0.412,0.78


**Observed:** base rate 0.531. Decay rate by bucket: <100 impressions 0.611 (n = 4,854), 100-499 0.558 (n = 5,141), 500-1,999 0.536 (n = 6,449), 2,000-9,999 0.513 (n = 6,573), 10,000+ 0.412 (n = 3,595). The n values add up to 26,612.

**Verdict: OPPOSITE.** The decay rate falls at every step up in volume (spread 19.9 points): higher-volume pages decayed less often, not more. A rule that favors high volume would therefore pick the pages least likely to be decaying. Why this happens was not tested.

### My rule, in plain words

**Rule (neglected low-volume page):** flag a page with 100 to 499 search impressions in 90 days that has not been updated for 91 days or more. Flagged pages rank higher the longer since the last update and the lower the volume. Other pages are not flagged.

**Why:** volume was OPPOSITE, so the rule looks at the low-volume side, and the 100-impression floor removes pages so small that a 30% change is a handful of impressions. Staleness was MIXED and weak, so it is only a 91-day cutoff and a way to order flagged pages.

**Score:** `min(days_since_last_update, 365) / 365 × (500 − impressions_90d) / 400` for flagged pages, 0 for the rest. No fitted weights.

**Reason code:** `neglected_low_volume`. **Action label:** `refresh_review` (flagged) or `no_action`. **Inputs:** only `days_since_last_update` and `impressions_90d`.

**Caveat:** the cutoffs were set after viewing the signal tables and are graded on the same data, so quality numbers will be somewhat optimistic.

## 2. Build the ranked queue (writes the CSV)

The rule from section 1, coded with no fitted weights. Every labeled page is ranked by score, highest first, with ties broken by `content_id` so the order is the same on every run. The queue is written to `work/outputs/baseline_action_score.csv` (kept out of git by design; the notebook regenerates it). It holds only the rank, page id, score, reason code, action label and the two rule inputs, with no label or not-allowed column.

In [5]:
import os

# Flag: enough traffic to judge, low volume, and not updated for 91+ days
flagged = ((df["impressions_90d"] >= 100) & (df["impressions_90d"] < 500)
           & (df["days_since_last_update"] >= 91))

stale_part = df["days_since_last_update"].clip(upper=365) / 365
volume_part = (500 - df["impressions_90d"]) / 400

df["score"] = np.where(flagged, stale_part * volume_part, 0.0)
df["reason_code"] = np.where(flagged, "neglected_low_volume", "none")
df["action_label"] = np.where(flagged, "refresh_review", "no_action")

# Rank everything; ties broken by content_id so the order never changes between runs
queue = df.sort_values(["score", "content_id"], ascending=[False, True]).reset_index(drop=True)
queue["rank"] = queue.index + 1

os.makedirs("work/outputs", exist_ok=True)
OUT = "work/outputs/baseline_action_score.csv"
cols = ["rank", "content_id", "score", "reason_code", "action_label",
        "days_since_last_update", "impressions_90d"]
queue[cols].to_csv(OUT, index=False)

print(f"Queue written: {OUT} ({len(queue):,} rows)")
print(f"Flagged pages: {int(flagged.sum()):,} ({flagged.mean():.1%})")
print(queue["action_label"].value_counts().to_string())
queue[cols].head(10)

Queue written: work/outputs/baseline_action_score.csv (26,612 rows)
Flagged pages: 1,531 (5.8%)
action_label
no_action         25081
refresh_review     1531


,rank,content_id,score,reason_code,action_label,days_since_last_update,impressions_90d
0,1,content_ab27c30d81f4,0.826630,neglected_low_volume,refresh_review,304,103
1,2,content_f488400fca67,0.720719,neglected_low_volume,refresh_review,305,155
2,3,content_02b0d6e30129,0.694603,neglected_low_volume,refresh_review,313,176
3,4,content_d25a099b3726,0.622534,neglected_low_volume,refresh_review,305,202
4,5,content_df1fa766cac2,0.612164,neglected_low_volume,refresh_review,304,206
5,6,content_e444c00065bd,0.508712,neglected_low_volume,refresh_review,211,148
6,7,content_b6e4581523ed,0.496356,neglected_low_volume,refresh_review,183,104
7,8,content_180fabd3a3e0,0.476301,neglected_low_volume,refresh_review,183,120
8,9,content_c6a9f1c16dee,0.462466,neglected_low_volume,refresh_review,211,180
9,10,content_6476d1d8c050,0.420192,neglected_low_volume,refresh_review,313,304


## 3. Top-10 review

For each of the top ten pages: the action, why it is there, and what would make it wrong. These lines use only the rule's logic and the observable page facts shown in the table below (CTR is in percent). The decay label was not consulted here; it is used for the weak-picks check in section 4.

| Rank | Page | Action | Why it is here | What would make it wrong |
|---|---|---|---|---|
| 1 | `content_ab27c30d81f4` | refresh_review | Highest score: 304 days since update and only 103 impressions, with 0 clicks at position 8.9 and a short 1,094 words. | 103 impressions may simply be this page's normal level, so a 30% drop would be about 30 impressions, which is noise. |
| 2 | `content_f488400fca67` | refresh_review | 305 days stale with 155 impressions and 0 clicks; short at 1,113 words. | It ranks near the top of page 1 (position 5.7), so low impressions may mean low demand, not decay, and a rewrite could disturb a good ranking. |
| 3 | `content_02b0d6e30129` | refresh_review | 313 days stale, 176 impressions, 0 clicks at position 6.9. | Its keyword has a search volume of only 110, so few impressions are expected; word count is missing, so length is unknown. |
| 4 | `content_d25a099b3726` | refresh_review | 305 days stale, 202 impressions, 0 clicks. | Average position is 64.5, so the page has barely been visible; it may never have performed, and "decay" would not apply. |
| 5 | `content_df1fa766cac2` | refresh_review | 304 days stale, 206 impressions, 1 click; short at 1,146 words. | At position 4.8 the page may be stable, and one click on 206 impressions is not evidence of a decline. |
| 6 | `content_e444c00065bd` | refresh_review | 211 days stale and only 148 impressions; short at 870 words. | Its CTR is 3.38%, far above the catalog median of about 0.1%, so it may be performing well for its size. |
| 7 | `content_b6e4581523ed` | refresh_review | 183 days stale, 104 impressions, 0 clicks at position 14.0. | Search volume is 0, so low impressions may just reflect absent demand; at 4,537 words the content itself is not obviously thin. |
| 8 | `content_180fabd3a3e0` | refresh_review | 183 days stale, 120 impressions, 1 click. | Position 6.0 on a long 4,633-word page suggests it may be healthy, and search volume is 0, so low demand is a likelier reason than decay. |
| 9 | `content_c6a9f1c16dee` | refresh_review | 211 days stale and 180 impressions, ranked here by the age and volume rule alone. | It looks like the healthiest of the ten: 21 clicks and an 11.67% CTR at position 3.7, so it may not need review. |
| 10 | `content_6476d1d8c050` | refresh_review | 313 days stale, 304 impressions, 0 clicks. | Average position is 67.8, so it has barely been visible; word count is missing, so the content cannot be judged. |

**Pattern across the ten:** most lines share one risk: the rule sees low volume and long staleness, but cannot tell a declining page from a page that is simply small. Rank 9 is the clearest doubtful pick, and ranks 4 and 10 point to a second problem: pages that barely ranked at all.

In [6]:
review_cols = ["rank", "content_id", "days_since_last_update", "impressions_90d",
               "clicks_90d", "ctr", "avg_position", "word_count", "search_volume"]
queue[review_cols].head(10)

,rank,content_id,days_since_last_update,impressions_90d,clicks_90d,ctr,avg_position,word_count,search_volume
0,1,content_ab27c30d81f4,304,103,0,0.00,8.9,1094.0,NaN
1,2,content_f488400fca67,305,155,0,0.00,5.7,1113.0,NaN
2,3,content_02b0d6e30129,313,176,0,0.00,6.9,NaN,110.0
3,4,content_d25a099b3726,305,202,0,0.00,64.5,NaN,10.0
4,5,content_df1fa766cac2,304,206,1,0.49,4.8,1146.0,NaN
5,6,content_e444c00065bd,211,148,5,3.38,16.6,870.0,NaN
6,7,content_b6e4581523ed,183,104,0,0.00,14.0,4537.0,0.0
7,8,content_180fabd3a3e0,183,120,1,0.83,6.0,4633.0,0.0
8,9,content_c6a9f1c16dee,211,180,21,11.67,3.7,872.0,NaN
9,10,content_6476d1d8c050,313,304,0,0.00,67.8,NaN,10.0


## 4. Weak picks + leakage check

**How the queue did** (the label is used here for grading only). Base rate 0.531. Precision of the top 10, 50, 100 and 500 pages was 0.500, 0.680, 0.640 and 0.618 (lift 0.94x, 1.28x, 1.20x, 1.16x); across all 1,531 flagged pages it was 0.591 (1.11x). The queue beats the base rate modestly, and the lift fades as the queue gets longer. The top 10 is too small to read: 5 of 10 decayed.

**Weak picks (5 of the top 10 did not decay):** ranks 1, 4, 6, 9 and 10.
- Ranks 4 and 10 barely ranked in search (average positions 64.5 and 67.8), so there was little visibility to lose.
- Ranks 6 and 9 look healthy (CTR 3.38% and 11.67%).
- Rank 1 has 103 impressions, so it may simply be small and steady.

The doubts written in section 3 fit these misses, but ranks 2, 3, 5, 7 and 8 carried doubts too and did decay, so those doubts do not tell which picks fail.

**What this says:** the rule cannot tell a declining small page from a small stable one, and it ignores whether a page ranks at all. Position and CTR are candidate signals for the Week-5 model; the rule is left unchanged here. The cutoffs were set on this same data, so these numbers are somewhat optimistic.

**Leakage check:** no rule input and no column in the CSV is on the not-allowed list, and the source data has no flag-like columns. The rule uses only `days_since_last_update` and `impressions_90d`; no label column or `*_last_30d` / `*_prev_30d` column is used. One caveat: `impressions_90d` is a trailing 90-day total that overlaps the 30-day windows defining the label. It is not label-derived, but the overlap may contribute to the volume pattern in signal 2, and this was not tested.

In [7]:
# 1) Does the queue beat the base rate? (the label is used for evaluation only)
base = queue["is_decayed"].mean()
flagged_q = queue[queue["action_label"] == "refresh_review"]

rows = [(f"top {k}", queue["is_decayed"].head(k).mean()) for k in [10, 50, 100, 500]]
rows.append((f"all {len(flagged_q):,} flagged", flagged_q["is_decayed"].mean()))
check = pd.DataFrame(rows, columns=["queue", "precision"])
check["lift_vs_base"] = (check["precision"] / base).round(2)
check["precision"] = check["precision"].round(3)
print(f"Base rate (all labeled pages): {base:.3f}")
print(check.to_string(index=False))

# 2) Which of the top ten actually decayed?
print()
print(queue[["rank", "content_id", "is_decayed"]].head(10).to_string(index=False))

# 3) Leakage check: no rule input or CSV column may be label-derived, and no product flags
RULE_INPUTS = ["days_since_last_update", "impressions_90d"]
CSV_COLUMNS = list(pd.read_csv(OUT, nrows=0).columns)
SOURCE_COLUMNS = pd.read_csv("data/raw/content_refresh_anonymized.csv", nrows=0).columns
print("\nRule inputs found in the not-allowed list:", sorted(set(RULE_INPUTS) & set(FORBIDDEN)))
print("CSV columns found in the not-allowed list: ", sorted(set(CSV_COLUMNS) & set(FORBIDDEN)))
print("Flag-like columns in the source data:      ",
      [c for c in SOURCE_COLUMNS if any(k in c.lower() for k in ["flag", "action", "recommend", "priority", "rule", "reason"])])

Base rate (all labeled pages): 0.531
            queue  precision  lift_vs_base
           top 10      0.500          0.94
           top 50      0.680          1.28
          top 100      0.640          1.20
          top 500      0.618          1.16
all 1,531 flagged      0.591          1.11

 rank           content_id  is_decayed
    1 content_ab27c30d81f4           0
    2 content_f488400fca67           1
    3 content_02b0d6e30129           1
    4 content_d25a099b3726           0
    5 content_df1fa766cac2           1
    6 content_e444c00065bd           0
    7 content_b6e4581523ed           1
    8 content_180fabd3a3e0           1
    9 content_c6a9f1c16dee           0
   10 content_6476d1d8c050           0

Rule inputs found in the not-allowed list: []
CSV columns found in the not-allowed list:  []
Flag-like columns in the source data:       []


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.